In [2]:
import os
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import lightgbm as lgb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import ndcg_score

In [ ]:
# train_data = pd.read_parquet("/Users/margherita/Desktop/processed_data/train.parquet")
# val_data = pd.read_parquet("/Users/margherita/Desktop/processed_data/val.parquet")
# test_data = pd.read_parquet("/Users/margherita/Desktop/processed_data/test.parquet")

DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"

train_data = pd.read_parquet(f"{DATA_DIR}/train_fe.parquet")
val_data = pd.read_parquet(f"{DATA_DIR}/val_fe.parquet")
test_data = pd.read_parquet(f"{DATA_DIR}/test_fe.parquet")

In [ ]:
train_data.head()

,srch_id,date_time,site_id,visitor_location_country_id,visitor_hist_starrating,visitor_hist_adr_usd,prop_country_id,prop_id,prop_starrating,prop_review_score,...,price_diff_vs_visitor_hist,prop_click_rate,prop_book_rate,prop_mean_position,prop_median_price,prop_n_impressions,dest_click_rate,dest_book_rate,dest_median_price,relevance
0,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,893,3,3.5,...,0.0,0.017826,0.740441,25.149622,118.000000,528,0.020658,0.763758,119.0,0
1,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,10404,4,4.0,...,0.0,0.021163,0.546149,22.723791,129.000000,496,0.020658,0.763758,119.0,0
2,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,21315,3,4.5,...,0.0,0.005889,0.462307,23.935345,168.920013,464,0.020658,0.763758,119.0,0
3,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,27348,2,4.0,...,0.0,0.016320,0.636885,24.353403,65.059998,382,0.020658,0.763758,119.0,0
4,1,2013-04-04 08:32:15,12,187,0.0,0.0,219,29604,4,3.5,...,0.0,0.031813,0.617864,12.907802,118.000000,564,0.020658,0.763758,119.0,0


In [ ]:
# Sort all datasets by search ID
# (all hotels belonging to the same search to be clustered together so it can compare them against each other)
train_data = train_data.sort_values('srch_id')
val_data = val_data.sort_values('srch_id')
test_data = test_data.sort_values('srch_id')

# Calculate the group sizes
# count exactly how many rows belong to each srch_id. So LightGBM knows where one search ends and the next begins.
train_groups = train_data.groupby('srch_id').size().values
val_groups = val_data.groupby('srch_id').size().values

print("Train group sizes:", train_groups)
print("Validation group sizes:", val_groups)

Train group sizes: [28 32  5 ... 24 28  6]
Validation group sizes: [28 29 34 ... 16  6 15]


In [ ]:
# Convert categorical features to category dtype and fill missing values with -1
categorical_features = ['site_id', 'visitor_location_country_id', 'prop_country_id', 'srch_destination_id']

for col in categorical_features:
    if col in train_data.columns and col in val_data.columns and col in test_data.columns:
        train_data[col] = train_data[col].astype(float).fillna(-1).astype(int).astype('category')
        val_data[col] = val_data[col].astype(float).fillna(-1).astype(int).astype('category')
        test_data[col] = test_data[col].astype(float).fillna(-1).astype(int).astype('category')

In [ ]:
# Define features and target
features = [col for col in train_data.columns if col not in ['srch_id', 'prop_id', 'relevance',
                                                             'date_time', 'click_bool', 'booking_bool',
                                                             'position', 'gross_bookings_usd']]
X_train = train_data[features]
y_train = train_data['relevance']
X_val = val_data[features]
y_val = val_data['relevance']
X_test = test_data[features]

In [ ]:
# Define the hyperparameter search space
param_grid = {
    'num_leaves': [20, 31, 63],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [100, 500, 1000],
}

n_iterations = 10
n_splits = 5

best_params = {}
best_cv_score = 0

X_full = X_train.copy()
y_full = y_train.copy()
groups_full = train_data['srch_id'].values

print("Starting hyperparameter tuning with GroupKFold cross-validation...")

for i in range(n_iterations):
    params = {key: np.random.choice(values) for key, values in param_grid.items()}
    print(f"Iteration {i+1}/{n_iterations}, Testing parameters: {params}")

    gkf = GroupKFold(n_splits=n_splits)
    fold_scores = []

    for train_idx, val_idx in gkf.split(X_full, y_full, groups=groups_full):
        X_train_fold, y_train_fold = X_full.iloc[train_idx], y_full.iloc[train_idx]
        X_val_fold, y_val_fold = X_full.iloc[val_idx], y_full.iloc[val_idx]

        train_fold_groups = X_train_fold.groupby(groups_full[train_idx]).size().values
        val_fold_groups = X_val_fold.groupby(groups_full[val_idx]).size().values

        ranker = lgb.LGBMRanker(objective='lambdarank', metric='ndcg', eval_at=[5], random_state=42, **params)
        ranker.fit(X_train_fold, y_train_fold, group=train_fold_groups,
            eval_set=[(X_val_fold, y_val_fold)],
            eval_group=[val_fold_groups],
            callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
        )

        best_iteration = ranker.best_iteration_
        fold_score = ranker.best_score_['valid_0']['ndcg@5']
        fold_scores.append(fold_score)

    avg_score = np.mean(fold_scores)
    print(f"Average NDCG@5 for iteration {i+1}: {avg_score:.4f}")
    if avg_score > best_cv_score:
        best_cv_score = avg_score
        best_params = params
        print(f"New best parameters found: {best_params} with NDCG@5: {best_cv_score:.4f}")

print(f"Best hyperparameters: {best_params} with NDCG@5: {best_cv_score:.4f}")

In [ ]:
best_params = {'num_leaves': np.int64(31), 'learning_rate': np.float64(0.1), 'n_estimators': np.int64(500)}
#Early stopping, best iteration is: [309]	valid_0's ndcg@5: 0.380935 with all features

In [ ]:
# Initialize the LGBMRanker with the specified parameters
ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    eval_at=[5],
    random_state=42,
    importance_type='gain',
    **best_params
)

# Fit the model with early stopping
ranker.fit(
    X_train, y_train,
    group=train_groups,
    eval_set=[(X_val, y_val)],
    eval_group=[val_groups],
    callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=50)]
)

In [ ]:
# After finding the best hyperparameters, we can train the final model on the combined training and validation data
X_full = pd.concat([X_train, X_val], ignore_index=True)
y_full = pd.concat([y_train, y_val], ignore_index=True)

groups_full = np.concatenate([train_groups, val_groups])

final_ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    eval_at=[5],
    random_state=42,
    importance_type='gain',
    n_estimators=350,
    num_leaves=31,
    learning_rate=0.1
)

print("Training final model on combined training and validation data...")
final_ranker.fit(
    X_full, y_full,
    group=groups_full,
    callbacks=[lgb.log_evaluation(period=100)]
)

In [ ]:
# Make predictions on the test set
X_test = test_data[features].copy()

# Force X_test to adopt the exact same categorical structure as X_full
for col in categorical_features:
    if col in X_test.columns:
        X_test[col] = X_test[col].astype(X_full[col].dtype)

test_data['predicted_score'] = final_ranker.predict(X_test)

# Sort the test data by Search ID, then by the predicted score descending
# This puts the highest scored hotels at the top for each search
test_predictions = test_data.sort_values(['srch_id', 'predicted_score'], ascending=[True, False])

# Keep only the necessary columns and rename them to match the required format[cite: 2]
submission = test_predictions[['srch_id', 'prop_id']].copy()
submission.columns = ['SearchId', 'PropertyId']

# Save to CSV without the index
submission.to_csv(f'{DATA_DIR}/submission_lightgbm.csv', index=False)
print("Submission saved to submission_lightgbm.csv!")

In [ ]:
import matplotlib.pyplot as plt

# Tell LightGBM to plot the top 20 most important features
lgb.plot_importance(final_ranker, max_num_features=20, figsize=(10, 8), importance_type='gain')
plt.title("LightGBM Feature Importance (Gain)")
# save the plot as a pdf
plt.savefig(f"{DATA_DIR}/feature_importance.pdf")
plt.show()